# 18 — Análise comercial integrada

**Transcrição → Processamento → Modelo → Indicadores → Recomendação**

Este é o ponto de entrada da solução. Ele recebe uma transcrição por vez, processa texto e termos, tenta os modelos disponíveis ou usa regras explícitas de fallback, calcula indicadores e sugere uma ação para revisão humana. A saída contém produto, sentimento, risco de churn, oportunidade, termos, mecanismo, scores e fontes em um único JSON.


## 1. Carregar os módulos

Os notebooks anteriores compartilham o mesmo kernel. Essa composição mantém as funcionalidades separadas sem criar uma biblioteca ou CLI paralela.

Execute esta célula após reiniciar o kernel. Ela carrega somente as células de definição dos notebooks anteriores; modelos pesados são inicializados durante a análise quando o modo permitir.


In [ ]:
from pathlib import Path

NOTEBOOKS_DIR = Path.cwd().resolve()
if not (NOTEBOOKS_DIR / '12_fundamentos_analise.ipynb').exists():
    NOTEBOOKS_DIR = NOTEBOOKS_DIR / 'notebooks'
if not (NOTEBOOKS_DIR / '12_fundamentos_analise.ipynb').exists():
    raise FileNotFoundError('Execute o notebook a partir da raiz ou da pasta notebooks.')

for module_name in [
    '12_fundamentos_analise.ipynb',
    '13_produtos_e_termos.ipynb',
    '14_analise_sentimento.ipynb',
    '15_risco_churn.ipynb',
    '16_oportunidade_comercial.ipynb',
    '17_recomendacao_acao.ipynb',
]:
    get_ipython().run_line_magic('run', f'"{NOTEBOOKS_DIR / module_name}"')


## 2. Carregar exatamente uma transcrição

`carregar_transcricao` aceita texto direto ou um arquivo, nunca os dois ao mesmo tempo. Arquivos `.txt` preservam o conteúdo original; arquivos `.json` devem conter um objeto e o campo configurado. Listas são rejeitadas para impedir processamento em lote acidental.

O texto direto e os arquivos são alternativas mutuamente exclusivas. A validação acontece antes de consultar modelos, e o conteúdo original será incluído literalmente em `transcricao_original`.


In [ ]:
def _validar_texto_transcricao(valor: Any, origem: str) -> str:
    if not isinstance(valor, str):
        raise TypeError(f"A transcrição de {origem} deve ser uma string.")
    if not valor.strip():
        raise ValueError(f"A transcrição de {origem} não pode estar vazia.")
    return valor


### Carregar uma transcrição

Escolha exatamente uma fonte. Para JSON, configure o nome do campo de texto; listas e campos ausentes geram erros claros antes da análise.


In [ ]:
def carregar_transcricao(
    *,
    texto: str | None = None,
    arquivo: str | Path | None = None,
    campo_json: str = "transcricao",
) -> str:
    """Carrega uma única transcrição sem alterar seu conteúdo."""
    if (texto is None) == (arquivo is None):
        raise ValueError("Informe exatamente uma fonte: texto ou arquivo.")
    if texto is not None:
        return _validar_texto_transcricao(texto, "texto direto")

    caminho = Path(arquivo)
    if not caminho.is_file():
        raise FileNotFoundError(f"Arquivo de transcrição não encontrado: {caminho}")
    if caminho.suffix.casefold() == ".txt":
        with caminho.open("r", encoding="utf-8", newline="") as stream:
            valor = stream.read()
    elif caminho.suffix.casefold() == ".json":
        try:
            payload = json.loads(caminho.read_text(encoding="utf-8"))
        except json.JSONDecodeError as error:
            raise ValueError(f"JSON inválido em {caminho.name}.") from error
        if not isinstance(payload, dict):
            raise ValueError("A entrada deve ser um objeto JSON com uma transcrição.")
        if campo_json not in payload:
            raise ValueError(f"O JSON não contém o campo {campo_json!r}.")
        valor = payload[campo_json]
    else:
        raise ValueError("Use um arquivo com extensão .txt ou .json.")
    return _validar_texto_transcricao(valor, caminho.name)


## 3. Persistência opcional

O resultado permanece apenas na memória quando o caminho de saída é `None`. Se um arquivo `.json` for configurado, a pasta é criada e o contrato completo é gravado em UTF-8.

O arquivo contém a transcrição integral e, por isso, deve ficar em local apropriado para dados sensíveis. Sem caminho configurado, a análise não é escrita no disco.


In [ ]:
def salvar_resultado(
    resultado: dict[str, Any], caminho_saida: str | Path | None
) -> Path | None:
    """Grava o JSON somente quando um caminho de saída é informado."""
    if caminho_saida is None:
        return None
    if not isinstance(resultado, dict):
        raise TypeError("O resultado deve ser um dicionário.")
    caminho = Path(caminho_saida)
    if caminho.suffix.casefold() != ".json":
        raise ValueError("O arquivo de saída deve usar a extensão .json.")
    caminho.parent.mkdir(parents=True, exist_ok=True)
    caminho.write_text(
        json.dumps(resultado, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    return caminho


## Como interpretar os scores

Os três tipos de score não são intercambiáveis. A legenda abaixo acompanha todo resultado, inclusive quando nenhum produto é encontrado; os metadados do produto também informam qual mecanismo realizou a busca.

Leia `score_type` junto com cada valor. `heuristic` expressa força de regras, `model_probability` representa a saída da classe do modelo e similaridade E5 mede proximidade textual.


In [ ]:
SCORE_LEGEND = {
    "model_probability": (
        "Confiança da label escolhida pelo modelo; não é calibrada para decisão comercial."
    ),
    "heuristic": (
        "Força relativa das evidências combinadas por regras explícitas; não representa probabilidade."
    ),
    "normalized_cosine_similarity": (
        "Similaridade semântica E5 normalizada entre zero e um; não representa probabilidade de compra."
    ),
}

### Metadados do produto

Mesmo sem candidatos, a saída registra qual retriever foi usado e qual tipo de score ele produziria.


In [ ]:
def _product_analysis_metadata(mode: str, has_candidates: bool) -> dict[str, Any]:
    uses_model = mode == "model"
    return {
        "engine": "multilingual_e5_small" if uses_model else "bm25_aliases",
        "model": E5_MODEL_NAME if uses_model else None,
        "score_type": "normalized_cosine_similarity" if uses_model else "heuristic",
        "status": "candidates_found" if has_candidates else "no_grounded_candidate",
    }


## 4. Contrato integrado

`analisar_transcricao` é a única interface que a pessoa usuária precisa conhecer. Ela valida a entrada, coordena os módulos e registra para cada indicador se houve uso de modelo ou fallback.

A execução reúne produtos, sentimento, risco de churn, oportunidade e termos antes de sugerir uma ação. As falhas de carregamento em `auto` aparecem em `analysis_mode.fallback_reasons`.


### Reunir motivos de fallback

Cada componente só registra uma causa quando tentou carregar um modelo no modo automático e precisou recorrer ao mecanismo lexical.


In [ ]:
def _analysis_fallback_reasons(
    product_reason: dict[str, str] | None,
    sentiment_reason: dict[str, str] | None,
    churn_reason: dict[str, str] | None,
    opportunity_reason: dict[str, str] | None,
) -> dict[str, dict[str, str]]:
    reasons = {}
    for component, reason in (
        ("products", product_reason), ("sentiment", sentiment_reason),
        ("churn", churn_reason), ("opportunity", opportunity_reason),
    ):
        if reason:
            reasons[component] = reason
    return reasons


### Produzir a análise comercial

Esta é a fronteira pública. A entrada é validada antes de carregar componentes; o contrato preserva o texto original e deixa explícitos scores, fontes e mecanismos usados.


In [ ]:
def analisar_transcricao(transcricao: str, modo: str = "auto") -> dict[str, Any]:
    """Analisa uma transcrição e devolve o contrato comercial da Sprint 4."""
    if not isinstance(transcricao, str):
        raise TypeError("A transcrição deve ser uma string.")
    if not transcricao.strip():
        raise ValueError("A transcrição não pode estar vazia.")
    if modo not in SUPPORTED_MODES:
        validos = ", ".join(sorted(SUPPORTED_MODES))
        raise ValueError(f"Modo inválido: {modo!r}. Use um de: {validos}.")

    produtos_candidatos, product_mode, product_reason = _analyze_products(
        transcricao, modo
    )
    sentimento, sentiment_mode, sentiment_reason = _analyze_sentiment(transcricao, modo)
    risco_churn, churn_mode, churn_reason = _analyze_churn(transcricao, modo)
    oportunidade, opportunity_mode, opportunity_reason = _analyze_opportunity(
        transcricao, modo
    )
    principais_termos = _extract_key_terms(transcricao)
    recomendacao = _recommend_action(
        produtos_candidatos, sentimento, risco_churn, oportunidade
    )
    return {
        "schema_version": "1.0",
        "transcricao_original": transcricao,
        "produto_identificado": (
            produtos_candidatos[0]["product"] if produtos_candidatos else None
        ),
        "produtos_candidatos": produtos_candidatos,
        "produto_metadados": _product_analysis_metadata(
            product_mode, bool(produtos_candidatos)
        ),
        "sentimento": sentimento,
        "risco_churn": risco_churn,
        "oportunidade_comercial": oportunidade,
        "principais_termos": principais_termos,
        "recomendacao_acao": recomendacao,
        "score_legend": SCORE_LEGEND,
        "analysis_mode": {
            "requested": modo,
            "components": {
                "products": product_mode,
                "sentiment": sentiment_mode,
                "churn": churn_mode,
                "opportunity": opportunity_mode,
            },
            "fallback_reasons": _analysis_fallback_reasons(
                product_reason, sentiment_reason, churn_reason, opportunity_reason
            ),
        },
    }


## 5. Configurar a entrada

Escolha `texto` para colar uma transcrição ou `arquivo` para ler `.txt`/`.json`. Em JSON, ajuste `CAMPO_JSON` ao nome que contém o texto, como `transcricao` ou `ANON_TRANSCRICAO`. Deixe `ARQUIVO_SAIDA=None` para não gravar o resultado.

Altere apenas os valores da próxima célula para uma nova execução. Use `MODO_ANALISE="fallback"` para uma demonstração local sem carregar modelos; `auto` tenta os artefatos disponíveis.


In [ ]:
FONTE_ENTRADA = "texto"  # texto ou arquivo
TRANSCRICAO_DIRETA = "Cole aqui a transcrição original da reunião."
ARQUIVO_ENTRADA = None  # exemplo: Path("data/minha_reuniao.txt")
CAMPO_JSON = "transcricao"
MODO_ANALISE = "auto"  # auto, full ou fallback
ARQUIVO_SAIDA = None  # exemplo: Path("data/processed/minha_analise.json")


## 6. Executar e revisar

O JSON mantém a transcrição original e os metadados necessários para auditar os indicadores. Nenhuma ação comercial deve ocorrer antes da revisão humana.

As etapas seguintes carregam, analisam, mostram as labels e só então gravam o JSON se `ARQUIVO_SAIDA` tiver sido preenchido. Confira o texto original, as fontes e os mecanismos antes de usar a sugestão.


### Carregar a fonte escolhida

Esta célula aplica a validação de fonte única. `transcricao` conserva inclusive espaços e quebras de linha do arquivo ou texto direto.


In [ ]:
if FONTE_ENTRADA == "texto":
    transcricao = carregar_transcricao(texto=TRANSCRICAO_DIRETA)
elif FONTE_ENTRADA == "arquivo":
    transcricao = carregar_transcricao(
        arquivo=ARQUIVO_ENTRADA, campo_json=CAMPO_JSON
    )
else:
    raise ValueError("FONTE_ENTRADA deve ser 'texto' ou 'arquivo'.")


### Analisar a transcrição

O modo escolhido determina quais modelos serão tentados. A função devolve o contrato inteiro em memória para inspeção antes de gravar um arquivo.


In [ ]:
import time

inicio_analise = time.perf_counter()
resultado = analisar_transcricao(transcricao, modo=MODO_ANALISE)
tempo_analise_segundos = time.perf_counter() - inicio_analise


### Resumo dos indicadores e das fontes

A tabela separa a label do tipo de score: probabilidade do modelo, similaridade de busca e score heurístico não têm a mesma interpretação. A fonte do produto vem do documento recuperado; para os demais indicadores, a evidência disponível é a própria transcrição. Confira especialmente se o produto principal é de fato o mencionado, pois documentos comparativos também podem aparecer como candidatos.


In [ ]:
from html import escape
from IPython.display import HTML, display

def _resumir_indicador(nome, indicador, fontes):
    score = indicador.get("score")
    score_texto = f"{score:.3f}" if isinstance(score, (int, float)) else "—"
    return (
        nome,
        str(indicador.get("label") or indicador.get("product") or "não identificado"),
        score_texto,
        str(indicador.get("score_type", "—")),
        str(indicador.get("engine", "—")),
        fontes,
    )

produto = resultado["produtos_candidatos"][0] if resultado["produtos_candidatos"] else {}
linhas = [
    _resumir_indicador("Produto", produto or resultado["produto_metadados"],
                       ", ".join(produto.get("sources", [])) or "sem fonte"),
    _resumir_indicador("Sentimento", resultado["sentimento"], "transcrição atual"),
    _resumir_indicador("Risco de churn", resultado["risco_churn"], "transcrição atual"),
    _resumir_indicador("Oportunidade", resultado["oportunidade_comercial"], "transcrição atual"),
]
cabecalhos = ("Indicador", "Label", "Score", "Tipo", "Mecanismo", "Fonte")
linhas_html = [cabecalhos, *linhas]
tabela_html = "<table>" + "".join(
    "<tr>" + "".join(
        f"<{('th' if indice == 0 else 'td')}>{escape(valor)}</{('th' if indice == 0 else 'td')}>"
        for valor in linha
    ) + "</tr>"
    for indice, linha in enumerate(linhas_html)
) + "</table>"
display(HTML(tabela_html))
print("Termos principais:", ", ".join(resultado["principais_termos"]))


### Conferir a recomendação

`criterios` registra as condições usadas pela regra de prioridade. `motivo` traduz essas condições; a sugestão continua dependendo da conferência das fontes e de revisão humana.


In [ ]:
acao = resultado["recomendacao_acao"]
print("Ação sugerida:", acao["label"])
print("Critérios:", ", ".join(acao["criterios"]))
print("Motivo:", acao["motivo"])
print("Revisão humana obrigatória:", acao["revisao_humana"])
print("Mecanismos utilizados:", resultado["analysis_mode"]["components"])
print("Motivos de fallback:", resultado["analysis_mode"]["fallback_reasons"])
print("Tempo da análise (s):", round(tempo_analise_segundos, 3))


### Inspecionar o JSON completo

O objeto abaixo inclui a transcrição original, fontes, scores e metadados. Evite compartilhar esta saída caso a transcrição contenha dados sensíveis.


In [ ]:
print(json.dumps(resultado, ensure_ascii=False, indent=2))


### Gravar somente se solicitado

Com `ARQUIVO_SAIDA=None`, esta etapa não persiste nada. Se houver um caminho `.json`, o arquivo conterá a transcrição original e deverá ser protegido como dado sensível.


In [ ]:
caminho_salvo = salvar_resultado(resultado, ARQUIVO_SAIDA)
if caminho_salvo is not None:
    print(f"Resultado salvo em: {caminho_salvo}")


## 7. Conclusões e limites da entrega

- **Contrato funcional:** `analisar_transcricao` recebe uma transcrição e retorna os indicadores e a recomendação. Os testes determinísticos verificam os ramos essenciais; o smoke test em 20 reuniões da base histórica, no modo `fallback`, teve zero falhas de contrato. Isso demonstra execução, não qualidade das labels.
- **Escolha para o protótipo:** BERTimbau ajustado para oportunidade e `multilingual-e5-small` para recuperação. Na pseudo-validação de oportunidade, BERTimbau obteve F1 de 0,974 e 3 erros em 140 chunks; TF-IDF + Regressão Logística obteve F1 de 0,951, recall maior (1,000 contra 0,983) e custo muito menor. A diferença de acertos não foi conclusiva (McNemar p = 0,375). Em 32 consultas curadas, E5 obteve Recall@5 de 0,984 e MRR de 0,922, superiores aos 0,938 e 0,766 de BM25 com aliases. Estes números vêm de `reports/metrics/model_comparison.json` e `reports/metrics/sentence_embeddings_retrieval.json`.
- **Sentimento e churn:** os modelos opcionais e as regras geram sinais para triagem. Não há avaliação de precisão dessas labels em reuniões comerciais anotadas por pessoas; scores de modelo não são probabilidades calibradas de churn ou compra.
- **Transcrições recebidas:** os agregados das 1.126 reuniões únicas coincidem com os da base histórica, mas não há prova de independência por ID. O smoke test de 20 reuniões não mede generalização para reuniões inéditas nem substitui rótulos humanos.
- **Próxima validação:** gerar/restaurar os artefatos BERTimbau e E5, executar o modo `full`, obter rótulos humanos e avaliar a solução em reuniões separadas por ID antes de ajustar limiares ou usar as ações sem supervisão. Veja `TODO.md` para os bloqueios e critérios.
